# LEGO Minifigure Classification - Consolidated Models Notebook

This is the single notebook for model training, evaluation, and reference code. The standalone training scripts were consolidated here so GitHub has one model entry point to manage.

## Repository Workflow

1. Install dependencies with `pip install -r requirements.txt`.
2. Place `minifigs.json` and the `images/` directory at the repository root. They are intentionally ignored by Git.
3. Run the cells for the model you want to train. The latest recommended implementation is **Option B V4**.
4. Generated checkpoints and result folders are ignored by Git; keep only summary documentation in the repository.

## Models Overview

| Model | Classes | Backbone | Purpose |
|---|---:|---|---|
| Baseline | 122 | EfficientNet-B0 | Raw-category reference baseline |
| Option B V4 | 28 | ConvNeXt-Small or EfficientNet-B4 | Latest optimized model with SWA and weighted ensemble |
| ResNet50 Top-20 | 20 | ResNet50 | Small reference baseline on the most common classes |
| Evaluation Utilities | Multiple | Saved checkpoints | Compare saved model outputs without retraining |

## Shared Notes

Most cells are copied from the previous standalone scripts. If running inside Jupyter, set `BASE_DIR` to the repository root when needed. Long-running training cells can take hours on CPU; GPU is recommended.

## Baseline: EfficientNet-B0 on 122 Raw Classes

Consolidated from `baseline_train.py`.

In [ ]:
"""
Baseline Model: Image-based LEGO Minifigure Category Classification
- Transfer learning with EfficientNet-B0
- Raw 122 categories (no merging)
- Stratified train/val/test split
- Full evaluation metrics
"""

import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, top_k_accuracy_score
)
from collections import Counter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = os.getcwd()
JSON_PATH = os.path.join(BASE_DIR, "minifigs.json")
IMG_DIR = os.path.join(BASE_DIR, "images")
OUTPUT_DIR = os.path.join(BASE_DIR, "baseline_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

IMG_SIZE = 128  # Smaller for faster baseline
BATCH_SIZE = 64
NUM_EPOCHS = 10
LR = 1e-3
NUM_WORKERS = 0  # MPS compatibility

# ============================================================
# LOAD DATA
# ============================================================
print("Loading data...")
with open(JSON_PATH) as f:
    data = json.load(f)

# Filter to records with existing images
valid_data = []
for d in data:
    if d.get('img_local_path'):
        img_path = os.path.join(BASE_DIR, d['img_local_path'])
        if os.path.exists(img_path):
            valid_data.append(d)

print(f"Total records: {len(data)}, Valid images: {len(valid_data)}")

# Build label encoding
categories = [d['category'] for d in valid_data]
cat_counts = Counter(categories)
cat_names = sorted(cat_counts.keys())
cat2idx = {c: i for i, c in enumerate(cat_names)}
idx2cat = {i: c for c, i in cat2idx.items()}
num_classes = len(cat_names)
print(f"Number of classes: {num_classes}")

# ============================================================
# STRATIFIED SPLIT (handle tiny classes)
# ============================================================
labels = [cat2idx[d['category']] for d in valid_data]

# Categories with <7 samples can't reliably stratify into 3 splits
# Put them all in train
label_counts = Counter(labels)
small_idx = [i for i, l in enumerate(labels) if label_counts[l] < 7]
big_idx = [i for i, l in enumerate(labels) if label_counts[l] >= 7]

big_data = [valid_data[i] for i in big_idx]
big_labels = [labels[i] for i in big_idx]

train_data_big, temp_data, train_labels_big, temp_labels = train_test_split(
    big_data, big_labels, test_size=0.3, random_state=42, stratify=big_labels
)
val_data, test_data, val_labels, test_labels = train_test_split(
    temp_data, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

# Add tiny-class samples to train
train_data = train_data_big + [valid_data[i] for i in small_idx]
train_labels = train_labels_big + [labels[i] for i in small_idx]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# ============================================================
# DATASET
# ============================================================
class MinifigDataset(Dataset):
    def __init__(self, records, labels, transform=None):
        self.records = records
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        img_path = os.path.join(BASE_DIR, self.records[idx]['img_local_path'])
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))

        if self.transform:
            image = self.transform(image)

        return image, self.labels[idx]

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = MinifigDataset(train_data, train_labels, train_transform)
val_dataset = MinifigDataset(val_data, val_labels, eval_transform)
test_dataset = MinifigDataset(test_data, test_labels, eval_transform)

# ============================================================
# CLASS WEIGHTS + SAMPLER
# ============================================================
train_label_counts = Counter(train_labels)
class_weights = torch.zeros(num_classes)
for i in range(num_classes):
    count = train_label_counts.get(i, 1)
    class_weights[i] = 1.0 / count
class_weights = class_weights / class_weights.sum() * num_classes
class_weights = class_weights.to(DEVICE)

# Weighted sampler for balanced batches
sample_weights = [1.0 / train_label_counts[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# ============================================================
# MODEL: EfficientNet-B0 (pretrained)
# ============================================================
print("Building model...")
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze early layers, fine-tune last blocks
for param in model.features[:6].parameters():
    param.requires_grad = False

# Replace classifier
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier[1].in_features, num_classes)
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

# ============================================================
# TRAINING LOOP
# ============================================================
print(f"\nTraining for {NUM_EPOCHS} epochs...")
print("-" * 65)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for images, labels_batch in train_loader:
        images, labels_batch = images.to(DEVICE), labels_batch.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        train_correct += preds.eq(labels_batch).sum().item()
        train_total += images.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for images, labels_batch in val_loader:
            images, labels_batch = images.to(DEVICE), labels_batch.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels_batch)

            val_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels_batch).sum().item()
            val_total += images.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
          f"LR: {lr:.6f} | {elapsed:.0f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

# ============================================================
# EVALUATION ON TEST SET
# ============================================================
print("\n" + "=" * 65)
print("EVALUATING ON TEST SET")
print("=" * 65)

model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pth"), weights_only=True))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels_batch in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        _, preds = outputs.max(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.numpy())
        all_probs.extend(probs)

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# ============================================================
# METRICS
# ============================================================
# Overall metrics
acc = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
macro_prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
macro_rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)

# Top-k accuracy
top3_acc = top_k_accuracy_score(all_labels, all_probs, k=3, labels=range(num_classes))
top5_acc = top_k_accuracy_score(all_labels, all_probs, k=5, labels=range(num_classes))

print(f"\n{'Metric':<30} {'Value':>10}")
print("-" * 42)
print(f"{'Accuracy':<30} {acc:>10.4f}")
print(f"{'Top-3 Accuracy':<30} {top3_acc:>10.4f}")
print(f"{'Top-5 Accuracy':<30} {top5_acc:>10.4f}")
print(f"{'Macro F1':<30} {macro_f1:>10.4f}")
print(f"{'Weighted F1':<30} {weighted_f1:>10.4f}")
print(f"{'Macro Precision':<30} {macro_prec:>10.4f}")
print(f"{'Macro Recall':<30} {macro_rec:>10.4f}")

# Per-class report
# Use explicit labels to handle missing classes in test set
present_labels = sorted(set(all_labels) | set(all_preds))
present_names = [idx2cat[i] for i in present_labels]

report = classification_report(
    all_labels, all_preds,
    labels=present_labels,
    target_names=present_names,
    zero_division=0,
    output_dict=True
)
report_str = classification_report(
    all_labels, all_preds,
    labels=present_labels,
    target_names=present_names,
    zero_division=0
)
print("\n" + report_str)

# Save report to file
with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), 'w') as f:
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"Top-3 Accuracy: {top3_acc:.4f}\n")
    f.write(f"Top-5 Accuracy: {top5_acc:.4f}\n")
    f.write(f"Macro F1: {macro_f1:.4f}\n")
    f.write(f"Weighted F1: {weighted_f1:.4f}\n\n")
    f.write(report_str)

# ============================================================
# PLOTS
# ============================================================

# 1. Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.close()

# 2. Confusion matrix (top-20 categories for readability)
top20_cats = [c for c, _ in Counter(all_labels).most_common(20)]
mask = np.isin(all_labels, top20_cats)
filtered_labels = all_labels[mask]
filtered_preds = all_preds[mask]

top20_names = [idx2cat[i] for i in top20_cats]
cm = confusion_matrix(filtered_labels, filtered_preds, labels=top20_cats)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=top20_names, yticklabels=top20_names, ax=ax,
            vmin=0, vmax=1)
ax.set_title('Normalized Confusion Matrix (Top 20 Categories)')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_top20.png"), dpi=150)
plt.close()

# 3. Per-class F1 bar chart
per_class_f1 = []
for cat in present_names:
    per_class_f1.append((cat, report[cat]['f1-score'], report[cat]['support']))

per_class_f1.sort(key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(16, 20))
names = [x[0] for x in per_class_f1]
f1s = [x[1] for x in per_class_f1]
supports = [x[2] for x in per_class_f1]
colors = ['green' if f > 0.5 else 'orange' if f > 0.2 else 'red' for f in f1s]
bars = ax.barh(range(len(names)), f1s, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels([f"{n} (n={s})" for n, s in zip(names, supports)], fontsize=7)
ax.set_xlabel('F1 Score')
ax.set_title('Per-Class F1 Score (green >0.5, orange >0.2, red ≤0.2)')
ax.invert_yaxis()
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "per_class_f1.png"), dpi=150)
plt.close()

# 4. Most confused pairs
print("\n" + "=" * 65)
print("TOP 20 MOST CONFUSED CATEGORY PAIRS")
print("=" * 65)

full_cm = confusion_matrix(all_labels, all_preds, labels=range(num_classes))
confused_pairs = []
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and full_cm[i][j] > 0:
            confused_pairs.append((idx2cat[i], idx2cat[j], full_cm[i][j]))

confused_pairs.sort(key=lambda x: -x[2])
print(f"{'True Category':<40} {'Predicted As':<40} {'Count':>5}")
print("-" * 87)
for true_c, pred_c, count in confused_pairs[:20]:
    print(f"{true_c:<40} {pred_c:<40} {count:>5}")

# 5. Categories with 0% F1
print("\n" + "=" * 65)
print("CATEGORIES WITH F1 = 0 (completely failed)")
print("=" * 65)
zero_f1 = [(n, s) for n, f, s in per_class_f1 if f == 0]
for n, s in zero_f1:
    print(f"  {n} (test samples: {s})")
print(f"\nTotal: {len(zero_f1)} categories with F1=0")

# 6. Summary by size bucket
print("\n" + "=" * 65)
print("PERFORMANCE BY CATEGORY SIZE")
print("=" * 65)
test_counts = Counter(all_labels)
buckets = {'<10': [], '10-49': [], '50-99': [], '100-299': [], '300+': []}
for cat in present_names:
    idx = cat2idx[cat]
    support = test_counts.get(idx, 0)
    f1 = report[cat]['f1-score']
    if support < 10: buckets['<10'].append(f1)
    elif support < 50: buckets['10-49'].append(f1)
    elif support < 100: buckets['50-99'].append(f1)
    elif support < 300: buckets['100-299'].append(f1)
    else: buckets['300+'].append(f1)

print(f"{'Size Bucket':<15} {'#Classes':>10} {'Avg F1':>10} {'Min F1':>10} {'Max F1':>10}")
print("-" * 57)
for bucket, f1s_list in buckets.items():
    if f1s_list:
        print(f"{bucket:<15} {len(f1s_list):>10} {np.mean(f1s_list):>10.4f} {min(f1s_list):>10.4f} {max(f1s_list):>10.4f}")

print(f"\nAll results saved to: {OUTPUT_DIR}")


## Latest Optimized Model: Option B V4

Consolidated from `optionB_v4_optimized.py`.

In [ ]:
"""
Option B V4 — Three targeted optimizations over V3:

  1. WEIGHTED MODEL ENSEMBLE
     V3 trains Focal / ArcFace / SupCon separately and picks only the best.
     The probability outputs of all three models are never combined, which
     leaves easy gains on the table.  We collect every model's softmax
     probabilities and average them with weights proportional to each
     model's validation F1 — a cheap but reliable diversity boost.

  2. STOCHASTIC WEIGHT AVERAGING (SWA)
     After the warm-up phase (first 75 % of epochs) we start averaging the
     model weights rather than just keeping the single best checkpoint.
     SWA finds a flatter minimum in the loss landscape, which typically
     improves generalisation by 1-3 % F1 with no extra data or parameters.
     PyTorch ships AveragedModel / SWALR / update_bn for this out of the box.

  3. CONVNEXT-SMALL BACKBONE (optional, flag below)
     ConvNeXt-Small was released after EfficientNet-B4 and consistently
     out-performs it on ImageNet at a similar parameter budget while being
     fully compatible with the torchvision transfer-learning workflow.
     Toggle USE_CONVNEXT = True to swap the backbone; everything else
     (head, losses, SWA, ensemble) stays identical.

  Everything not listed above is intentionally unchanged from V3 so that
  the delta in results is attributable to these three changes only.
"""

import json, os, time, warnings, random, math, copy
warnings.filterwarnings('ignore')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler  # Mixed precision training
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn   # <-- SWA
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, top_k_accuracy_score,
)
from collections import Counter
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR   = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "optionB_v4_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device(
    "cuda"  if torch.cuda.is_available() else
    "mps"   if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else
    "cpu"
)
print(f"Using device: {DEVICE}")

# ── Backbone choice ──────────────────────────────────────────────────────────
# False → EfficientNet-B4 (same as V3, good for direct comparison)
# True  → ConvNeXt-Small  (new in V4, typically +1-2 % F1 at same cost)
USE_CONVNEXT = True

IMG_SIZE        = 384 if USE_CONVNEXT else 380   # ConvNeXt prefers 384
BATCH_SIZE      = 16
NUM_EPOCHS      = 20
LR              = 3e-4
NUM_WORKERS     = 4
PATIENCE        = 5
LABEL_SMOOTHING = 0.1
FOCAL_GAMMA     = 2.0
MIXUP_ALPHA     = 0.2
CUTMIX_ALPHA    = 1.0
CUTMIX_PROB     = 0.3
MIXUP_PROB      = 0.3
MIN_SAMPLES_TARGET = 50

# SWA starts after this fraction of epochs has elapsed
SWA_START_FRAC = 0.75          # begin averaging at epoch ⌊0.75 × NUM_EPOCHS⌋
SWA_LR         = 5e-5          # constant LR used during SWA phase

backbone_name = "ConvNeXt-Small" if USE_CONVNEXT else "EfficientNet-B4"
print(f"Backbone: {backbone_name}  |  IMG_SIZE: {IMG_SIZE}  |  SWA start: {SWA_START_FRAC:.0%}")

# ─────────────────────────────────────────────────────────────────────────────
# TOWN SPLITTER & MERGE MAP  (identical to V3)
# ─────────────────────────────────────────────────────────────────────────────
def split_town(subcategory):
    sub = subcategory.lower()
    if 'police'      in sub: return 'Town - Police'
    if 'fire'        in sub: return 'Town - Fire'
    if 'airport'     in sub: return 'Town - Airport'
    if 'hospital'    in sub or 'rescue'      in sub: return 'Town - Rescue'
    if 'space'       in sub: return 'Town - Space'
    if 'race'        in sub or 'stuntz'      in sub: return 'Town - Racing'
    if 'coast guard' in sub: return 'Town - Coast Guard'
    if 'construction'in sub: return 'Town - Construction'
    if any(x in sub for x in ['arctic','jungle','volcano','ocean','deep sea','exploration']):
        return 'Town - Exploration'
    return 'Town - General'

MERGE_MAP = {
    'The LEGO NINJAGO Movie': 'NINJAGO',
    'The LEGO Movie': 'LEGO Movies', 'The LEGO Movie 2': 'LEGO Movies',
    'DC Super Hero Girls': 'Super Heroes', 'Spider-Man': 'Super Heroes', 'Batman I': 'Super Heroes',
    'Elves': 'Friends & Fantasy', 'Friends': 'Friends & Fantasy',
    'DUPLO': 'Preschool', 'Primo': 'Preschool', 'Belville': 'Preschool', 'Scala': 'Preschool',
    'For Juniors': 'Preschool', 'Homemaker': 'Preschool', 'Fabuland': 'Preschool',
    'Pirates of the Caribbean': 'Pirates',
    'Castle': 'Castle & Medieval', 'Vikings': 'Castle & Medieval', 'Ninja': 'Castle & Medieval',
    'NEXO KNIGHTS': 'Castle & Medieval',
    'Space': 'Space & Sci-Fi', 'Aquazone': 'Space & Sci-Fi', 'Alpha Team': 'Space & Sci-Fi',
    'Exo-Force': 'Space & Sci-Fi', 'Power Miners': 'Space & Sci-Fi', 'Rock Raiders': 'Space & Sci-Fi',
    'Agents': 'Space & Sci-Fi', 'Ultra Agents': 'Space & Sci-Fi', 'Atlantis': 'Space & Sci-Fi',
    'Adventurers': 'Adventure', 'Indiana Jones': 'Adventure', "Pharaoh's Quest": 'Adventure',
    'Dino Attack': 'Adventure', 'Dino': 'Adventure', 'Western': 'Adventure',
    'SPEED CHAMPIONS': 'Racing & Vehicles', 'Racers': 'Racing & Vehicles',
    'World Racers': 'Racing & Vehicles', 'Speed Racer': 'Racing & Vehicles', 'Cars': 'Racing & Vehicles',
    'LEGENDS OF CHIMA': 'Fantasy', 'DREAMZzz': 'Fantasy', 'Hidden Side': 'Fantasy',
    'Monster Fighters': 'Fantasy',
    'The Hobbit and The Lord of the Rings': 'Middle-Earth',
    'Teenage Mutant Ninja Turtles': 'Licensed Entertainment',
    'SpongeBob SquarePants': 'Licensed Entertainment', 'Scooby-Doo': 'Licensed Entertainment',
    'The Simpsons': 'Licensed Entertainment', 'Ghostbusters': 'Licensed Entertainment',
    'Stranger Things': 'Licensed Entertainment', 'The Angry Birds Movie': 'Licensed Entertainment',
    'Trolls World Tour': 'Licensed Entertainment', 'The Incredibles': 'Licensed Entertainment',
    'Toy Story': 'Licensed Entertainment', 'Wednesday': 'Licensed Entertainment',
    'Wicked': 'Licensed Entertainment', 'Despicable Me and Minions': 'Licensed Entertainment',
    'The Lone Ranger': 'Licensed Entertainment', 'Prince of Persia': 'Licensed Entertainment',
    'Island Xtreme Stunts': 'Licensed Entertainment', 'The Powerpuff Girls': 'Licensed Entertainment',
    'Back to the Future': 'Licensed Entertainment', 'Friends TV Series': 'Licensed Entertainment',
    'Queer Eye': 'Licensed Entertainment', 'Lightyear': 'Licensed Entertainment',
    'Bluey': 'Licensed Entertainment', "Gabby's Dollhouse": 'Licensed Entertainment',
    'Dune': 'Licensed Entertainment', 'Star Trek': 'Licensed Entertainment',
    'Overwatch': 'Video Games', 'Fortnite': 'Video Games', 'Sonic the Hedgehog': 'Video Games',
    'Horizon': 'Video Games', 'The Legend of Zelda': 'Video Games', 'Animal Crossing': 'Video Games',
    'Avatar The Last Airbender': 'Video Games', 'Avatar': 'Video Games', 'One Piece': 'Video Games',
    'Collectible Minifigures': 'Collectible & Special', 'Holiday & Event': 'Collectible & Special',
    'Dimensions': 'Collectible & Special', 'Vidiyo': 'Collectible & Special',
    'Games': 'Collectible & Special',
    'LEGO Ideas': 'LEGO Promotional', 'BrickLink Designer Program': 'LEGO Promotional',
    'LEGO Brand': 'LEGO Promotional', 'LEGOLAND': 'LEGO Promotional',
    'LEGOLAND Parks': 'LEGO Promotional', 'Promotional': 'LEGO Promotional',
    'Studios': 'LEGO Promotional', 'FIRST LEGO League': 'LEGO Promotional',
    'Educational & Dacta': 'Education & Technical', 'BIONICLE': 'Education & Technical',
    'Hero Factory': 'Education & Technical', 'Technic': 'Education & Technical',
    'Basic': 'Education & Technical', 'FreeStyle': 'Education & Technical',
    'Master Builder Academy': 'Education & Technical', 'Building Bigger Thinking': 'Education & Technical',
    'Discovery': 'Education & Technical', 'Quatro': 'Education & Technical',
    'Universe': 'Education & Technical', 'Fusion': 'Education & Technical',
    'Nike': 'Education & Technical', 'Architecture': 'Education & Technical',
    'Clikits': 'Education & Technical', 'Unikitty!': 'Education & Technical',
}

# ─────────────────────────────────────────────────────────────────────────────
# LOSS FUNCTIONS  (unchanged from V3)
# ─────────────────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight, self.gamma, self.label_smoothing = weight, gamma, label_smoothing

    def forward(self, inputs, targets):
        n = inputs.size(1)
        log_probs = F.log_softmax(inputs, dim=1)
        probs = torch.exp(log_probs)
        if self.label_smoothing > 0:
            smooth = torch.full_like(inputs, self.label_smoothing / (n - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
            fw = (1 - probs).pow(self.gamma)
            loss = -(fw * smooth * log_probs).sum(dim=1)
        else:
            ce = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
            pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
            loss = (1 - pt).pow(self.gamma) * ce
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()


class ArcFaceHead(nn.Module):
    def __init__(self, in_features, num_classes, scale=30.0, margin=0.5):
        super().__init__()
        self.scale, self.margin = scale, margin
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m, self.sin_m = math.cos(margin), math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, emb, labels=None):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight))
        if labels is None:
            return cosine * self.scale
        sine  = torch.sqrt(1.0 - torch.clamp(cosine * cosine, 0, 1))
        phi   = cosine * self.cos_m - sine * self.sin_m
        phi   = torch.where(cosine > self.th, phi, cosine - self.mm)
        oh    = torch.zeros_like(cosine).scatter_(1, labels.view(-1, 1), 1)
        return (oh * phi + (1 - oh) * cosine) * self.scale


class ArcFaceLoss(nn.Module):
    def __init__(self, weight=None, label_smoothing=0.1):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight, label_smoothing=label_smoothing)
    def forward(self, logits, targets): return self.ce(logits, targets)


class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device, B = features.device, features.shape[0]
        labels = labels.contiguous().view(-1, 1)
        mask   = torch.eq(labels, labels.T).float().to(device)
        adot   = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(adot, dim=1, keepdim=True)
        logits = adot - logits_max.detach()
        lmask  = 1 - torch.eye(B, device=device)
        mask   = mask * lmask
        exp_l  = torch.exp(logits) * lmask
        log_prob = logits - torch.log(exp_l.sum(1, keepdim=True) + 1e-12)
        mp = mask.sum(1).clamp(min=1)
        return -(mask * log_prob).sum(1).div(mp).mean()

# ─────────────────────────────────────────────────────────────────────────────
# AUGMENTATION HELPERS  (unchanged from V3)
# ─────────────────────────────────────────────────────────────────────────────
def rand_bbox(size, lam):
    H, W = size[2], size[3]
    cr = np.sqrt(1.0 - lam)
    ch, cw = int(H * cr), int(W * cr)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2 = np.clip(cy - ch // 2, 0, H), np.clip(cy + ch // 2, 0, H)
    x1, x2 = np.clip(cx - cw // 2, 0, W), np.clip(cx + cw // 2, 0, W)
    return y1, y2, x1, x2

def apply_cutmix(imgs, tgts, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0)).to(imgs.device)
    y1, y2, x1, x2 = rand_bbox(imgs.size(), lam)
    imgs[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (imgs.size(-1) * imgs.size(-2))
    return imgs, tgts, tgts[idx], lam

def apply_mixup(imgs, tgts, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0)).to(imgs.device)
    return lam * imgs + (1 - lam) * imgs[idx], tgts, tgts[idx], lam


class PadToSquare:
    def __call__(self, img):
        w, h = img.size
        return ImageOps.pad(img, (max(w, h), max(w, h)), color=(255, 255, 255))

# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print("\nLoading data …")
json_path = os.path.join(BASE_DIR, "minifigs.json")
with open(json_path) as f:
    raw = json.load(f)

valid_data = [d for d in raw
              if d.get('img_local_path')
              and os.path.exists(os.path.join(BASE_DIR, d['img_local_path']))]

for d in valid_data:
    if d['category'] == 'Town':
        d['merged_category'] = split_town(d.get('subcategory', ''))
    else:
        d['merged_category'] = MERGE_MAP.get(d['category'], d['category'])

categories  = [d['merged_category'] for d in valid_data]
cat_counts  = Counter(categories)
cat_names   = sorted(cat_counts.keys())
cat2idx     = {c: i for i, c in enumerate(cat_names)}
idx2cat     = {i: c for c, i in cat2idx.items()}
num_classes = len(cat_names)
print(f"Valid images: {len(valid_data)},  Classes: {num_classes}")

# ─────────────────────────────────────────────────────────────────────────────
# STRATIFIED SPLIT  (identical to V3)
# ─────────────────────────────────────────────────────────────────────────────
labels      = [cat2idx[d['merged_category']] for d in valid_data]
label_counts= Counter(labels)
small_idx   = [i for i, l in enumerate(labels) if label_counts[l] < 7]
big_idx     = [i for i, l in enumerate(labels) if label_counts[l] >= 7]

big_data, big_labels = [valid_data[i] for i in big_idx], [labels[i] for i in big_idx]
train_big, temp_data, train_labels_big, temp_labels = train_test_split(
    big_data, big_labels, test_size=0.30, random_state=42, stratify=big_labels)
val_data, test_data, val_labels, test_labels = train_test_split(
    temp_data, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels)

train_data   = train_big   + [valid_data[i] for i in small_idx]
train_labels = train_labels_big + [labels[i] for i in small_idx]
print(f"Train: {len(train_data)},  Val: {len(val_data)},  Test: {len(test_data)}")

# ─────────────────────────────────────────────────────────────────────────────
# TRANSFORMS  (10 TTA views — V3 only had 5)
# ─────────────────────────────────────────────────────────────────────────────
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    PadToSquare(),
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
    transforms.Normalize(mean, std),
])

eval_transform = transforms.Compose([
    PadToSquare(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# V4: 10 TTA views (V3 had 5)
# More diversity → better probability calibration when ensembling
tta_transforms = [
    # 1. standard
    eval_transform,
    # 2. horizontal flip
    transforms.Compose([PadToSquare(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize(mean, std)]),
    # 3. slight zoom-in
    transforms.Compose([PadToSquare(), transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
                        transforms.CenterCrop(IMG_SIZE),
                        transforms.ToTensor(), transforms.Normalize(mean, std)]),
    # 4. more zoom-in
    transforms.Compose([PadToSquare(), transforms.Resize((IMG_SIZE + 40, IMG_SIZE + 40)),
                        transforms.CenterCrop(IMG_SIZE),
                        transforms.ToTensor(), transforms.Normalize(mean, std)]),
    # 5. colour jitter
    transforms.Compose([PadToSquare(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.ColorJitter(brightness=0.15, contrast=0.15),
                        transforms.ToTensor(), transforms.Normalize(mean, std)]),
    # 6. small rotation CW
    transforms.Compose([PadToSquare(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.RandomRotation(degrees=(5, 10)),
                        transforms.ToTensor(), transforms.Normalize(mean, std)]),
    # 7. small rotation CCW
    transforms.Compose([PadToSquare(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.RandomRotation(degrees=(-10, -5)),
                        transforms.ToTensor(), transforms.Normalize(mean, std)]),
    # 8. flip + zoom
    transforms.Compose([PadToSquare(), transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
                        transforms.CenterCrop(IMG_SIZE),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize(mean, std)]),
    # 9. slight brightness increase
    transforms.Compose([PadToSquare(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.ColorJitter(brightness=(1.1, 1.3)),
                        transforms.ToTensor(), transforms.Normalize(mean, std)]),
    # 10. slight brightness decrease
    transforms.Compose([PadToSquare(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.ColorJitter(brightness=(0.7, 0.9)),
                        transforms.ToTensor(), transforms.Normalize(mean, std)]),
]
print(f"TTA views: {len(tta_transforms)}  (V3 had 5)")

# ─────────────────────────────────────────────────────────────────────────────
# DATASET & LOADERS
# ─────────────────────────────────────────────────────────────────────────────
class MinifigDataset(Dataset):
    def __init__(self, records, labels, transform):
        self.records, self.labels, self.tf = records, labels, transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        try:
            p = self.records[idx]['img_local_path']
            p = p if os.path.isabs(p) else os.path.join(BASE_DIR, p)
            img = Image.open(p).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (255, 255, 255))
        return self.tf(img), self.labels[idx]


class EmbeddingDataset(Dataset):
    """Two-view dataset for SupCon stage-1."""
    def __init__(self, records, labels, transform):
        self.records, self.labels, self.tf = records, labels, transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        try:
            p = self.records[idx]['img_local_path']
            p = p if os.path.isabs(p) else os.path.join(BASE_DIR, p)
            img = Image.open(p).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (255, 255, 255))
        return self.tf(img), self.tf(img), self.labels[idx]


# Class weights — same sqrt formula as V3
tlc = Counter(train_labels)
class_weights = torch.zeros(num_classes)
for i in range(num_classes):
    class_weights[i] = 1.0 / math.sqrt(tlc.get(i, 1))
class_weights = (class_weights / class_weights.sum() * num_classes).to(DEVICE)

sample_weights = [1.0 / math.sqrt(tlc[l]) for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_dataset = MinifigDataset(train_data,  train_labels, train_transform)
val_dataset   = MinifigDataset(val_data,    val_labels,   eval_transform)
test_dataset  = MinifigDataset(test_data,   test_labels,  eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# ─────────────────────────────────────────────────────────────────────────────
# MODEL BUILDER
# ─────────────────────────────────────────────────────────────────────────────
def build_model(loss_type='focal'):
    """
    Returns (model, arcface_head_or_None, embedding_dim).
    Supports EfficientNet-B4 (V3 default) or ConvNeXt-Small (V4 option).
    """
    if USE_CONVNEXT:
        # ── ConvNeXt-Small ───────────────────────────────────────────────────
        from torchvision.models import convnext_small, ConvNeXt_Small_Weights
        backbone_raw  = convnext_small(weights=ConvNeXt_Small_Weights.IMAGENET1K_V1)
        in_feat       = 768  # ConvNeXt-Small feature dimension

        # Wrap to include global pooling + strip final linear layers
        class ConvNeXtBackbone(nn.Module):
            def __init__(self, model):
                super().__init__()
                self.features = model.features
                self.avgpool = model.avgpool
            def forward(self, x):
                x = self.features(x)
                x = self.avgpool(x)
                return x.flatten(1)

        backbone = ConvNeXtBackbone(backbone_raw)

        # Freeze first 2 stages (out of 4)
        stage_params = list(backbone_raw.features[:2].parameters())
        for p in stage_params:
            p.requires_grad = False
    else:
        # ── EfficientNet-B4  (same as V3, via torchvision) ──────────────────
        backbone = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        in_feat  = backbone.classifier[1].in_features    # 1792
        backbone.classifier = nn.Identity()

        # Freeze early blocks (conv_stem + blocks 0-3)
        for name, p in backbone.named_parameters():
            if name.startswith('features.0') or \
               any(name.startswith(f'features.{i}') for i in range(1, 5)):
                p.requires_grad = False

    arcface_head = None

    if loss_type == 'arcface':
        head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_feat, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(0.2),
        )
        arcface_head = ArcFaceHead(512, num_classes, scale=30.0, margin=0.5).to(DEVICE)
        model = nn.Sequential(backbone, head)
    elif loss_type == 'supcon':
        head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_feat, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
        )
        model = nn.Sequential(backbone, head)
    else:   # focal / default
        head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_feat, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )
        model = nn.Sequential(backbone, head)

    model = model.to(DEVICE)
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Params: {total:,} total | {trainable:,} trainable ({trainable/total*100:.1f}%)")
    return model, arcface_head, in_feat

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING FUNCTION  — key V4 addition: SWA
# ─────────────────────────────────────────────────────────────────────────────
def train_model(model, loss_type, arcface_head=None, supcon_stage1=False,
                num_epochs=NUM_EPOCHS):
    """
    Train with optional SWA.  SWA is activated after SWA_START_FRAC × epochs.
    Returns (best_val_f1, history, swa_model_or_None).
    """
    swa_start_epoch = int(SWA_START_FRAC * num_epochs)

    # Loss functions
    if loss_type == 'focal':
        criterion = FocalLoss(weight=class_weights, gamma=FOCAL_GAMMA,
                              label_smoothing=LABEL_SMOOTHING)
    elif loss_type == 'arcface':
        criterion = ArcFaceLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    elif loss_type == 'supcon' and supcon_stage1:
        criterion = SupConLoss(temperature=0.07)
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)

    criterion_val = nn.CrossEntropyLoss()

    params = list(model.parameters()) + (list(arcface_head.parameters()) if arcface_head else [])
    optimizer = torch.optim.AdamW(
        [p for p in params if p.requires_grad], lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2)

    # ── Mixed precision training ─────────────────────────────────────────────
    scaler = GradScaler() if torch.cuda.is_available() else None
    # ─────────────────────────────────────────────────────────────────────────

    # ── SWA setup ────────────────────────────────────────────────────────────
    swa_model     = AveragedModel(model)          # wraps the base model
    swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=3)
    swa_active    = False
    # ─────────────────────────────────────────────────────────────────────────

    history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc','val_f1']}
    best_val_f1, patience_ctr = 0.0, 0
    save_path = os.path.join(OUTPUT_DIR, f"best_model_{loss_type}.pth")

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        if arcface_head: arcface_head.train()
        tl, tc, tt = 0.0, 0, 0

        # ── Stage-1 SupCon  ──────────────────────────────────────────────────
        if loss_type == 'supcon' and supcon_stage1:
            ds  = EmbeddingDataset(train_data, train_labels, train_transform)
            ldr = DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                             num_workers=NUM_WORKERS)
            for v1, v2, lbl in ldr:
                v1, v2, lbl = v1.to(DEVICE), v2.to(DEVICE), lbl.to(DEVICE)
                optimizer.zero_grad()
                with autocast(enabled=(DEVICE.type == 'cuda')):
                    z1 = F.normalize(model(v1), dim=1)
                    z2 = F.normalize(model(v2), dim=1)
                    loss = criterion(torch.cat([z1, z2]), torch.cat([lbl, lbl]))
                if scaler is not None:
                    scaler.scale(loss).backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimizer.step()
                tl += loss.item() * v1.size(0); tt += v1.size(0)
            tl /= max(tt, 1); ta = 0.0
        # ── Standard training ────────────────────────────────────────────────
        else:
            for imgs, lbl in train_loader:
                imgs, lbl = imgs.to(DEVICE), lbl.to(DEVICE)
                r = random.random()
                optimizer.zero_grad()

                with autocast(enabled=(DEVICE.type == 'cuda')):
                    if r < CUTMIX_PROB:
                        imgs, ta_, tb_, lam = apply_cutmix(imgs, lbl, CUTMIX_ALPHA)
                        out = arcface_head(model(imgs), ta_) if arcface_head else model(imgs)
                        ob  = arcface_head(model(imgs), tb_) if arcface_head else out
                        loss = lam * criterion(out, ta_) + (1-lam) * criterion(ob, tb_)
                    elif r < CUTMIX_PROB + MIXUP_PROB:
                        imgs, ta_, tb_, lam = apply_mixup(imgs, lbl, MIXUP_ALPHA)
                        out = arcface_head(model(imgs), ta_) if arcface_head else model(imgs)
                        ob  = arcface_head(model(imgs), tb_) if arcface_head else out
                        loss = lam * criterion(out, ta_) + (1-lam) * criterion(ob, tb_)
                    else:
                        out = arcface_head(model(imgs), lbl) if arcface_head else model(imgs)
                        loss = criterion(out, lbl)

                if scaler is not None:
                    scaler.scale(loss).backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimizer.step()

                tl += loss.item() * imgs.size(0)
                tc += out.max(1)[1].eq(lbl).sum().item()
                tt += imgs.size(0)
            tl /= max(tt, 1); ta = tc / max(tt, 1)

        # ── SWA update / LR schedule ─────────────────────────────────────────
        if epoch >= swa_start_epoch and not (loss_type == 'supcon' and supcon_stage1):
            swa_model.update_parameters(model)
            swa_scheduler.step()
            if not swa_active:
                print(f"  [SWA activated at epoch {epoch+1}]")
                swa_active = True
        else:
            scheduler.step(epoch)
        # ─────────────────────────────────────────────────────────────────────

        # ── Validation ───────────────────────────────────────────────────────
        model.eval()
        if arcface_head: arcface_head.eval()
        vl, vc, vt = 0.0, 0, 0
        vp_all, vl_all = [], []

        if not (loss_type == 'supcon' and supcon_stage1):
            with torch.no_grad():
                for imgs, lbl in val_loader:
                    imgs, lbl = imgs.to(DEVICE), lbl.to(DEVICE)
                    out = arcface_head(model(imgs)) if arcface_head else model(imgs)
                    vl += criterion_val(out, lbl).item() * imgs.size(0)
                    vc += out.max(1)[1].eq(lbl).sum().item()
                    vt += imgs.size(0)
                    vp_all.extend(out.max(1)[1].cpu().numpy())
                    vl_all.extend(lbl.cpu().numpy())
            vl /= max(vt, 1); va = vc / max(vt, 1)
            val_f1 = f1_score(vl_all, vp_all, average='macro', zero_division=0)
        else:
            vl = va = val_f1 = 0.0

        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        history['val_f1'].append(val_f1)

        lr_now  = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0
        marker  = ""

        if not (loss_type == 'supcon' and supcon_stage1):
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                torch.save({'model': model.state_dict(),
                            **(({'arcface': arcface_head.state_dict()}) if arcface_head else {})},
                           save_path)
                patience_ctr = 0; marker = "  ← best"
            else:
                patience_ctr += 1

            swa_tag = " [SWA]" if swa_active else ""
            print(f"  Ep {epoch+1:02d}/{num_epochs}{swa_tag} | "
                  f"trn {tl:.4f}/{ta:.4f}  val {vl:.4f}/{va:.4f}  F1 {val_f1:.4f} | "
                  f"lr {lr_now:.2e} | {elapsed:.0f}s{marker}")

            if patience_ctr >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break
        else:
            torch.save(model.state_dict(), save_path)
            print(f"  Ep {epoch+1:02d}/{num_epochs} (SupCon S1) | loss {tl:.4f} | "
                  f"lr {lr_now:.2e} | {elapsed:.0f}s")

    # ── Finalise SWA model ───────────────────────────────────────────────────
    if swa_active:
        print("  Updating BatchNorm statistics for SWA model …")
        update_bn(train_loader, swa_model, device=DEVICE)
        swa_save = save_path.replace(".pth", "_swa.pth")
        torch.save(swa_model.state_dict(), swa_save)
        print(f"  SWA model saved → {swa_save}")
    # ─────────────────────────────────────────────────────────────────────────

    return best_val_f1, history, swa_model if swa_active else None


# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION HELPER  (supports both a plain model and an SWA-wrapped model)
# ─────────────────────────────────────────────────────────────────────────────
def predict_probs(model, arcface_head=None):
    """Run TTA over the test set and return (labels, averaged_probs)."""
    model.eval()
    if arcface_head: arcface_head.eval()

    def one_pass(loader):
        pbs, lbs = [], []
        with torch.no_grad():
            for imgs, lbl in loader:
                imgs = imgs.to(DEVICE)
                out  = arcface_head(model(imgs)) if arcface_head else model(imgs)
                pbs.extend(torch.softmax(out, 1).cpu().numpy())
                lbs.extend(lbl.numpy())
        return np.array(lbs), np.array(pbs)

    labels, base_probs = one_pass(test_loader)

    print(f"  TTA ({len(tta_transforms)} views) …")
    acc_probs = np.zeros_like(base_probs)
    for t in tta_transforms:
        ds  = MinifigDataset(test_data, test_labels, t)
        ldr = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        _, pb = one_pass(ldr)
        acc_probs += pb
    acc_probs /= len(tta_transforms)

    return labels, acc_probs


def metrics_from_probs(labels, probs):
    preds = probs.argmax(1)
    return {
        'Accuracy':        accuracy_score(labels, preds),
        'Macro F1':        f1_score(labels, preds, average='macro',    zero_division=0),
        'Weighted F1':     f1_score(labels, preds, average='weighted', zero_division=0),
        'Macro Precision': precision_score(labels, preds, average='macro', zero_division=0),
        'Macro Recall':    recall_score(labels,    preds, average='macro', zero_division=0),
        'Top-3 Accuracy':  top_k_accuracy_score(labels, probs, k=3, labels=range(num_classes)),
        'Top-5 Accuracy':  top_k_accuracy_score(labels, probs, k=5, labels=range(num_classes)),
    }


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN & EVALUATE ALL THREE VARIANTS
# ─────────────────────────────────────────────────────────────────────────────
all_probs   = {}   # loss_type → test-set probability matrix
all_val_f1  = {}   # loss_type → best validation F1 (used for ensemble weights)
all_metrics = {}   # loss_type → metrics dict

# ── 1. FOCAL ─────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("TRAINING  V4-Focal")
print("═"*70)
m_focal, _, _ = build_model('focal')
bf_focal, hist_focal, swa_focal = train_model(m_focal, 'focal')
ck = torch.load(os.path.join(OUTPUT_DIR, "best_model_focal.pth"), map_location=DEVICE, weights_only=True)
m_focal.load_state_dict(ck.get('model', ck))
print("  Evaluating best checkpoint …")
labels_ref, probs_focal = predict_probs(m_focal)
all_probs['focal']  = probs_focal
all_val_f1['focal'] = bf_focal
all_metrics['focal'] = metrics_from_probs(labels_ref, probs_focal)

# Also evaluate the SWA model if it was produced
if swa_focal is not None:
    print("  Evaluating SWA model …")
    _, probs_focal_swa = predict_probs(swa_focal)
    all_probs['focal_swa']   = probs_focal_swa
    all_val_f1['focal_swa']  = bf_focal          # same val F1 (use checkpoint val)
    all_metrics['focal_swa'] = metrics_from_probs(labels_ref, probs_focal_swa)

del m_focal, swa_focal
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# ── 2. ARCFACE ────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("TRAINING  V4-ArcFace")
print("═"*70)
m_arc, af_head, _ = build_model('arcface')
bf_arc, hist_arc, swa_arc = train_model(m_arc, 'arcface', arcface_head=af_head)
ck = torch.load(os.path.join(OUTPUT_DIR, "best_model_arcface.pth"), map_location=DEVICE, weights_only=True)
m_arc.load_state_dict(ck.get('model', ck))
if 'arcface' in ck: af_head.load_state_dict(ck['arcface'])
print("  Evaluating best checkpoint …")
_, probs_arc = predict_probs(m_arc, arcface_head=af_head)
all_probs['arcface']  = probs_arc
all_val_f1['arcface'] = bf_arc
all_metrics['arcface'] = metrics_from_probs(labels_ref, probs_arc)

del m_arc, af_head, swa_arc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# ── 3. SUPCON ─────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("TRAINING  V4-SupCon  (stage 1 = contrastive  →  stage 2 = CE fine-tune)")
print("═"*70)
supcon_epochs = max(5, NUM_EPOCHS // 4)   # stage-1 epochs (shorter)
m_supcon, _, in_feat = build_model('supcon')
print(f"  Stage 1: contrastive ({supcon_epochs} epochs)")
_, hist_s1, _ = train_model(m_supcon, 'supcon', supcon_stage1=True, num_epochs=supcon_epochs)

# Swap projection head → classification head for stage 2
backbone_only = m_supcon[0]
cls_head = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_feat, 512), nn.BatchNorm1d(512), nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, num_classes),
).to(DEVICE)
m_supcon_s2 = nn.Sequential(backbone_only, cls_head).to(DEVICE)
print(f"  Stage 2: CE fine-tune ({NUM_EPOCHS} epochs)")
bf_supcon, hist_s2, swa_supcon = train_model(m_supcon_s2, 'supcon_s2')

ck_path = os.path.join(OUTPUT_DIR, "best_model_supcon_s2.pth")
if os.path.exists(ck_path):
    ck = torch.load(ck_path, map_location=DEVICE, weights_only=True)
    m_supcon_s2.load_state_dict(ck.get('model', ck))
print("  Evaluating best checkpoint …")
_, probs_supcon = predict_probs(m_supcon_s2)
all_probs['supcon']  = probs_supcon
all_val_f1['supcon'] = bf_supcon
all_metrics['supcon'] = metrics_from_probs(labels_ref, probs_supcon)

del m_supcon, m_supcon_s2, swa_supcon
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# ─────────────────────────────────────────────────────────────────────────────
# V4 KEY ADDITION: WEIGHTED ENSEMBLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("COMPUTING WEIGHTED ENSEMBLE  (V4 key optimisation)")
print("═"*70)

# Use the 3 primary models (Focal, ArcFace, SupCon) for the ensemble.
# Weight each model's probabilities by its validation Macro-F1 score so that
# the stronger model contributes more to the final prediction.
ensemble_keys = ['focal', 'arcface', 'supcon']
raw_weights   = np.array([all_val_f1[k] for k in ensemble_keys])
norm_weights  = raw_weights / raw_weights.sum()

print("  Ensemble members and their weights:")
for k, w, f in zip(ensemble_keys, norm_weights, raw_weights):
    print(f"    {k:<12}  val-F1={f:.4f}  →  weight={w:.4f}")

ensemble_probs = sum(norm_weights[i] * all_probs[ensemble_keys[i]]
                     for i in range(len(ensemble_keys)))
all_probs['ensemble']   = ensemble_probs
all_metrics['ensemble'] = metrics_from_probs(labels_ref, ensemble_probs)

# Uniform ensemble as a sanity-check baseline
uniform_probs = sum(all_probs[k] for k in ensemble_keys) / len(ensemble_keys)
all_probs['ensemble_uniform']   = uniform_probs
all_metrics['ensemble_uniform'] = metrics_from_probs(labels_ref, uniform_probs)

print(f"\n  Weighted ensemble Macro F1 : {all_metrics['ensemble']['Macro F1']:.4f}")
print(f"  Uniform  ensemble Macro F1 : {all_metrics['ensemble_uniform']['Macro F1']:.4f}")

# If we produced SWA models, include them in a "super-ensemble" as well
if 'focal_swa' in all_probs:
    se_keys = ensemble_keys + ['focal_swa']
    se_w    = np.array([all_val_f1[k] for k in se_keys])
    se_w   /= se_w.sum()
    se_probs = sum(se_w[i] * all_probs[se_keys[i]] for i in range(len(se_keys)))
    all_probs['super_ensemble']   = se_probs
    all_metrics['super_ensemble'] = metrics_from_probs(labels_ref, se_probs)
    print(f"  Super-ensemble (+ SWA) Macro F1 : {all_metrics['super_ensemble']['Macro F1']:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# RESULTS TABLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("FULL COMPARISON")
print("═"*70)

# Historical V3 reference numbers (from RESULTS.md)
v3_ref = {
    'V3-Focal':  {'Macro F1': 0.7736, 'Accuracy': None, 'Weighted F1': None},
    'V3-SupCon': {'Macro F1': 0.7951, 'Accuracy': None, 'Weighted F1': None},
}

metrics_order = ['Accuracy','Macro F1','Weighted F1','Macro Precision',
                 'Macro Recall','Top-3 Accuracy','Top-5 Accuracy']

header = f"{'Model':<24}" + "".join(f"{m:>18}" for m in metrics_order)
print(header)
print("-" * len(header))

for ref_name, ref_vals in v3_ref.items():
    row = f"{ref_name + ' (V3 ref)':<24}"
    for m in metrics_order:
        v = ref_vals.get(m)
        row += f"{'—':>18}" if v is None else f"{v:>18.4f}"
    print(row)

print("-" * len(header))
for name, mets in all_metrics.items():
    row = f"{'V4-'+name:<24}"
    for m in metrics_order:
        row += f"{mets.get(m, 0):>18.4f}"
    print(row)

# ─────────────────────────────────────────────────────────────────────────────
# CLASSIFICATION REPORT  (best model)
# ─────────────────────────────────────────────────────────────────────────────
best_key = max(all_metrics, key=lambda k: all_metrics[k]['Macro F1'])
best_probs = all_probs[best_key]
best_preds = best_probs.argmax(1)

print(f"\nBest model: V4-{best_key}  (Macro F1 = {all_metrics[best_key]['Macro F1']:.4f})")
report = classification_report(labels_ref, best_preds,
                                target_names=[idx2cat[i] for i in range(num_classes)],
                                zero_division=0)
print(report)

report_path = os.path.join(OUTPUT_DIR, f"classification_report_{best_key}.txt")
with open(report_path, 'w') as f:
    f.write(f"Best model: V4-{best_key}\n\n")
    f.write(f"Macro F1 : {all_metrics[best_key]['Macro F1']:.4f}\n")
    f.write(f"Accuracy : {all_metrics[best_key]['Accuracy']:.4f}\n\n")
    f.write(report)
print(f"Saved → {report_path}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────────────────────

# 1. Training curves (Focal — representative)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist_focal['train_loss'], label='train'); axes[0].plot(hist_focal['val_loss'], label='val')
axes[0].set_title('Focal — Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(hist_focal['val_f1'])
axes[1].set_title('Focal — Val Macro F1'); axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves_focal.png"), dpi=150)
plt.close()

# 2. Confusion matrix (best model)
cm   = confusion_matrix(labels_ref, best_preds)
cmn  = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(max(12, num_classes // 2), max(10, num_classes // 2)))
sns.heatmap(cmn, annot=False, fmt='.2f', xticklabels=[idx2cat[i] for i in range(num_classes)],
            yticklabels=[idx2cat[i] for i in range(num_classes)], ax=ax, cmap='Blues')
ax.set_title(f'Normalised Confusion Matrix — V4-{best_key}')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"confusion_matrix_{best_key}.png"), dpi=150)
plt.close()
print(f"Saved confusion matrix → {OUTPUT_DIR}/confusion_matrix_{best_key}.png")

# 3. Per-class F1 bar chart
per_cls_f1 = f1_score(labels_ref, best_preds, average=None, zero_division=0)
order       = np.argsort(per_cls_f1)
fig, ax     = plt.subplots(figsize=(10, max(8, num_classes * 0.3)))
ax.barh([idx2cat[i] for i in order], per_cls_f1[order], color='steelblue')
ax.axvline(per_cls_f1.mean(), color='red', linestyle='--', label=f'mean={per_cls_f1.mean():.3f}')
ax.set_xlabel('F1 Score'); ax.set_title(f'Per-Class F1 — V4-{best_key}'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"per_class_f1_{best_key}.png"), dpi=150)
plt.close()

# 4. Ensemble comparison bar chart
ens_names = list(all_metrics.keys())
ens_f1    = [all_metrics[k]['Macro F1'] for k in ens_names]
fig, ax   = plt.subplots(figsize=(10, 5))
bars = ax.bar(ens_names, ens_f1, color=['steelblue' if 'ensemble' not in k else 'darkorange'
                                         for k in ens_names])
ax.axhline(0.7951, color='green', linestyle='--', label='V3-SupCon best (0.7951)')
for b, v in zip(bars, ens_f1):
    ax.text(b.get_x() + b.get_width()/2, v + 0.003, f'{v:.4f}', ha='center', fontsize=8)
ax.set_ylabel('Macro F1'); ax.set_title('V4 Model Comparison (with Ensemble)'); ax.legend()
ax.set_xticklabels(ens_names, rotation=25, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ensemble_comparison.png"), dpi=150)
plt.close()
print(f"Saved ensemble comparison → {OUTPUT_DIR}/ensemble_comparison.png")

# ─────────────────────────────────────────────────────────────────────────────
# SAMPLE PREDICTIONS  (grid: image | true | predicted | confidence)
# ─────────────────────────────────────────────────────────────────────────────
print("\nGenerating sample-prediction grid …")
rng  = np.random.default_rng(0)
idxs = rng.choice(len(test_data), size=16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(18, 18))
for ax, idx in zip(axes.flat, idxs):
    p  = os.path.join(BASE_DIR, test_data[idx]['img_local_path'])
    img = Image.open(p).convert('RGB') if os.path.exists(p) else Image.new('RGB',(100,100),(200,200,200))
    pred_idx   = best_preds[idx]
    true_idx   = labels_ref[idx]
    confidence = best_probs[idx, pred_idx]
    correct    = pred_idx == true_idx
    ax.imshow(img)
    ax.axis('off')
    color = 'green' if correct else 'red'
    ax.set_title(f"True: {idx2cat[true_idx]}\nPred: {idx2cat[pred_idx]}\nConf: {confidence:.2f}",
                 fontsize=7, color=color)
plt.suptitle(f'Sample Predictions — V4-{best_key}', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_predictions.png"), dpi=150)
plt.close()
print(f"Saved sample predictions → {OUTPUT_DIR}/sample_predictions.png")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY PRINTOUT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("V4 OPTIMISATION SUMMARY")
print("═"*70)
print(f"Backbone:            {backbone_name}")
print(f"SWA:                 activated at epoch {int(SWA_START_FRAC*NUM_EPOCHS)+1} / {NUM_EPOCHS}")
print(f"TTA views:           {len(tta_transforms)}  (V3 had 5)")
print(f"Ensemble members:    {', '.join(ensemble_keys)}")
print()
print("Per-model test Macro F1:")
for k in ensemble_keys:
    delta = all_metrics[k]['Macro F1'] - 0.7951   # vs V3 SupCon best
    sign  = '+' if delta >= 0 else ''
    print(f"  V4-{k:<14} {all_metrics[k]['Macro F1']:.4f}  ({sign}{delta:.4f} vs V3 best)")
print()
print("Ensemble test Macro F1:")
for k in [kk for kk in all_metrics if 'ensemble' in kk]:
    delta = all_metrics[k]['Macro F1'] - 0.7951
    sign  = '+' if delta >= 0 else ''
    print(f"  V4-{k:<20} {all_metrics[k]['Macro F1']:.4f}  ({sign}{delta:.4f} vs V3 best)")

print(f"\nAll artefacts saved to: {OUTPUT_DIR}/")
print("Done.")


## Reference Model: ResNet50 Top-20 Classes

Consolidated from `resnet50_simple.py`.

In [ ]:
"""
Simple ResNet50 Fine-tuning on Top 20 Classes
- ResNet50 pretrained backbone
- Top 20 classes only (78.9% of data = 13.7K samples)
- Standard cross-entropy loss
- Two-stage training: frozen backbone → full fine-tuning
- ~30 min training on GPU, ~2-3 hours on CPU
"""

import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from collections import Counter
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "resnet50_top20_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

IMG_SIZE = 224  # ResNet standard
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS_FROZEN = 5
NUM_EPOCHS_FULL = 15
NUM_WORKERS = 4

# Top 20 classes
TOP_20_CLASSES = [
    'Town', 'Star Wars', 'Super Heroes', 'DUPLO', 'NINJAGO',
    'Friends', 'Collectible Minifigures', 'Harry Potter', 'Castle', 'Holiday & Event',
    'Disney', 'LEGO Ideas', 'BrickLink Designer Program', 'Minecraft', 'Sports',
    'Super Mario', 'Space', 'Pirates', 'Monkie Kid', 'LEGO Brand'
]

print(f"Training on {len(TOP_20_CLASSES)} classes: {', '.join(TOP_20_CLASSES)}")

# ─────────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
class MinifigDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
        except:
            # Fallback: create gray image if loading fails
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=128)

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx]

def load_data():
    """Load minifigs.json and filter to top 20 classes."""
    print("Loading data...")
    with open(os.path.join(BASE_DIR, 'minifigs.json')) as f:
        minifigs = json.load(f)

    # Filter to top 20 classes
    class_to_idx = {cls: i for i, cls in enumerate(TOP_20_CLASSES)}

    image_paths = []
    labels = []
    skipped = 0

    for entry in minifigs:
        category = entry.get('category')
        if category not in class_to_idx:
            continue

        img_local_path = entry.get('img_local_path')
        if not img_local_path:
            skipped += 1
            continue

        img_path = os.path.join(BASE_DIR, img_local_path)
        if not os.path.exists(img_path):
            skipped += 1
            continue

        image_paths.append(img_path)
        labels.append(class_to_idx[category])

    print(f"Loaded {len(image_paths)} images, skipped {skipped}")
    print(f"Class distribution: {Counter(labels)}")

    return image_paths, labels, class_to_idx

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if (batch_idx + 1) % 50 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

def evaluate(model, loader, criterion, device):
    """Evaluate on validation set."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    return avg_loss, acc, f1, np.array(all_preds), np.array(all_labels)

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    start_time = time.time()

    # Load data
    image_paths, labels, class_to_idx = load_data()

    # Train/val split
    X_train, X_val, y_train, y_val = train_test_split(
        image_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )
    print(f"Train: {len(X_train)}, Val: {len(X_val)}")

    # Data augmentation
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    # Datasets
    train_dataset = MinifigDataset(X_train, y_train, train_transform)
    val_dataset = MinifigDataset(X_val, y_val, val_transform)

    # Weighted sampler for class balance
    class_counts = Counter(y_train)
    weights = [1.0 / class_counts[label] for label in y_train]
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # Model: ResNet50 pretrained
    print("\nLoading ResNet50 pretrained on ImageNet...")
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, len(TOP_20_CLASSES))
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    # Phase 1: Frozen backbone, train only head (5 epochs)
    print("\n" + "="*70)
    print("PHASE 1: Frozen backbone - optimizing fc layer only (5 epochs)")
    print("="*70)

    # Only optimize the fc (head) parameters
    optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)
    best_val_f1 = 0.0

    for epoch in range(NUM_EPOCHS_FROZEN):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS_FROZEN}")
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, DEVICE)

        print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model.pth'))
            print(f"✓ Best model saved (F1={val_f1:.4f})")

    # Phase 2: Full fine-tuning (15 epochs)
    print("\n" + "="*70)
    print("PHASE 2: Full fine-tuning - all parameters (15 epochs)")
    print("="*70)

    # Now optimize all parameters with lower learning rate
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE * 0.1)  # Lower LR for full training

    for epoch in range(NUM_EPOCHS_FULL):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS_FULL}")
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, DEVICE)

        print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model.pth'))
            print(f"✓ Best model saved (F1={val_f1:.4f})")

    # Final evaluation on validation set
    print("\n" + "="*70)
    print("FINAL EVALUATION")
    print("="*70)

    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth')))
    _, _, _, val_preds, val_labels = evaluate(model, val_loader, criterion, DEVICE)

    # Classification report
    report = classification_report(val_labels, val_preds, target_names=TOP_20_CLASSES, zero_division=0)
    print(report)

    with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(report)

    # Confusion matrix
    cm = confusion_matrix(val_labels, val_preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, xticklabels=TOP_20_CLASSES, yticklabels=TOP_20_CLASSES,
                cmap='Blues', annot=False, cbar=True)
    plt.title('Confusion Matrix - ResNet50 Top 20 Classes')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
    print(f"\n✓ Confusion matrix saved")

    # Per-class F1 scores
    per_class_f1 = f1_score(val_labels, val_preds, average=None, zero_division=0)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(range(len(TOP_20_CLASSES)), per_class_f1, color='steelblue')
    ax.set_xticks(range(len(TOP_20_CLASSES)))
    ax.set_xticklabels(TOP_20_CLASSES, rotation=45, ha='right')
    ax.set_ylabel('F1 Score')
    ax.set_title('Per-Class F1 Scores')
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'per_class_f1.png'), dpi=150)
    print(f"✓ Per-class F1 chart saved")

    # Summary
    elapsed = time.time() - start_time
    acc = accuracy_score(val_labels, val_preds)
    macro_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(val_labels, val_preds, average='weighted', zero_division=0)

    summary = f"""
ResNet50 Fine-tuning Results
============================
Classes: {len(TOP_20_CLASSES)}
Total samples: {len(X_train) + len(X_val)}
Training time: {elapsed/60:.1f} minutes

Final Validation Metrics:
- Accuracy: {acc:.4f}
- Macro F1: {macro_f1:.4f}
- Weighted F1: {weighted_f1:.4f}

Per-class F1 scores:
"""
    for cls, f1 in zip(TOP_20_CLASSES, per_class_f1):
        summary += f"\n  {cls:30s}: {f1:.4f}"

    print(summary)

    with open(os.path.join(OUTPUT_DIR, 'results_summary.txt'), 'w') as f:
        f.write(summary)

    print(f"\n✓ All results saved to {OUTPUT_DIR}/")

if __name__ == '__main__':
    main()


## Evaluation Utilities: Compare Saved Models

Consolidated from `evaluate_all.py`.

In [ ]:
"""Evaluate all 4 trained models on the test set (no retraining)."""
import json, os, warnings, math
warnings.filterwarnings('ignore')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, top_k_accuracy_score
)
from collections import Counter
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = os.getcwd()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
NUM_WORKERS = 2

# ============================================================
# MERGE MAPS
# ============================================================
MERGE_MAP_B = {
    'The LEGO NINJAGO Movie': 'NINJAGO',
    'The LEGO Movie': 'LEGO Movies', 'The LEGO Movie 2': 'LEGO Movies',
    'DC Super Hero Girls': 'Super Heroes', 'Spider-Man': 'Super Heroes', 'Batman I': 'Super Heroes',
    'Elves': 'Friends & Fantasy', 'Friends': 'Friends & Fantasy',
    'DUPLO': 'Preschool', 'Primo': 'Preschool', 'Belville': 'Preschool', 'Scala': 'Preschool',
    'For Juniors': 'Preschool', 'Homemaker': 'Preschool', 'Fabuland': 'Preschool',
    'Pirates of the Caribbean': 'Pirates',
    'Castle': 'Castle & Medieval', 'Vikings': 'Castle & Medieval', 'Ninja': 'Castle & Medieval',
    'NEXO KNIGHTS': 'Castle & Medieval',
    'Space': 'Space & Sci-Fi', 'Aquazone': 'Space & Sci-Fi', 'Alpha Team': 'Space & Sci-Fi',
    'Exo-Force': 'Space & Sci-Fi', 'Power Miners': 'Space & Sci-Fi', 'Rock Raiders': 'Space & Sci-Fi',
    'Agents': 'Space & Sci-Fi', 'Ultra Agents': 'Space & Sci-Fi', 'Atlantis': 'Space & Sci-Fi',
    'Adventurers': 'Adventure', 'Indiana Jones': 'Adventure', "Pharaoh's Quest": 'Adventure',
    'Dino Attack': 'Adventure', 'Dino': 'Adventure', 'Western': 'Adventure',
    'SPEED CHAMPIONS': 'Racing & Vehicles', 'Racers': 'Racing & Vehicles',
    'World Racers': 'Racing & Vehicles', 'Speed Racer': 'Racing & Vehicles', 'Cars': 'Racing & Vehicles',
    'LEGENDS OF CHIMA': 'Fantasy', 'DREAMZzz': 'Fantasy', 'Hidden Side': 'Fantasy',
    'Monster Fighters': 'Fantasy',
    'The Hobbit and The Lord of the Rings': 'Middle-Earth',
    'Teenage Mutant Ninja Turtles': 'Licensed Entertainment',
    'SpongeBob SquarePants': 'Licensed Entertainment', 'Scooby-Doo': 'Licensed Entertainment',
    'The Simpsons': 'Licensed Entertainment', 'Ghostbusters': 'Licensed Entertainment',
    'Stranger Things': 'Licensed Entertainment', 'The Angry Birds Movie': 'Licensed Entertainment',
    'Trolls World Tour': 'Licensed Entertainment', 'The Incredibles': 'Licensed Entertainment',
    'Toy Story': 'Licensed Entertainment', 'Wednesday': 'Licensed Entertainment',
    'Wicked': 'Licensed Entertainment', 'Despicable Me and Minions': 'Licensed Entertainment',
    'The Lone Ranger': 'Licensed Entertainment', 'Prince of Persia': 'Licensed Entertainment',
    'Island Xtreme Stunts': 'Licensed Entertainment', 'The Powerpuff Girls': 'Licensed Entertainment',
    'Back to the Future': 'Licensed Entertainment', 'Friends TV Series': 'Licensed Entertainment',
    'Queer Eye': 'Licensed Entertainment', 'Lightyear': 'Licensed Entertainment',
    'Bluey': 'Licensed Entertainment', "Gabby's Dollhouse": 'Licensed Entertainment',
    'Dune': 'Licensed Entertainment', 'Star Trek': 'Licensed Entertainment',
    'Overwatch': 'Video Games', 'Fortnite': 'Video Games', 'Sonic the Hedgehog': 'Video Games',
    'Horizon': 'Video Games', 'The Legend of Zelda': 'Video Games', 'Animal Crossing': 'Video Games',
    'Avatar The Last Airbender': 'Video Games', 'Avatar': 'Video Games', 'One Piece': 'Video Games',
    'Collectible Minifigures': 'Collectible & Special', 'Holiday & Event': 'Collectible & Special',
    'Dimensions': 'Collectible & Special', 'Vidiyo': 'Collectible & Special',
    'Games': 'Collectible & Special',
    'LEGO Ideas': 'LEGO Promotional', 'BrickLink Designer Program': 'LEGO Promotional',
    'LEGO Brand': 'LEGO Promotional', 'LEGOLAND': 'LEGO Promotional',
    'LEGOLAND Parks': 'LEGO Promotional', 'Promotional': 'LEGO Promotional',
    'Studios': 'LEGO Promotional', 'FIRST LEGO League': 'LEGO Promotional',
    'Educational & Dacta': 'Education & Technical', 'BIONICLE': 'Education & Technical',
    'Hero Factory': 'Education & Technical', 'Technic': 'Education & Technical',
    'Basic': 'Education & Technical', 'FreeStyle': 'Education & Technical',
    'Master Builder Academy': 'Education & Technical', 'Building Bigger Thinking': 'Education & Technical',
    'Discovery': 'Education & Technical', 'Quatro': 'Education & Technical',
    'Universe': 'Education & Technical', 'Fusion': 'Education & Technical',
    'Nike': 'Education & Technical', 'Architecture': 'Education & Technical',
    'Clikits': 'Education & Technical', 'Unikitty!': 'Education & Technical',
}

def split_town(subcategory):
    sub = subcategory.lower()
    if 'police' in sub: return 'Town - Police'
    if 'fire' in sub: return 'Town - Fire'
    if 'airport' in sub: return 'Town - Airport'
    if 'hospital' in sub or 'rescue' in sub: return 'Town - Rescue'
    if 'space' in sub: return 'Town - Space'
    if 'race' in sub or 'stuntz' in sub: return 'Town - Racing'
    if 'coast guard' in sub: return 'Town - Coast Guard'
    if 'construction' in sub: return 'Town - Construction'
    if any(x in sub for x in ['arctic', 'jungle', 'volcano', 'ocean', 'deep sea', 'exploration']): return 'Town - Exploration'
    return 'Town - General'

# ============================================================
# LOAD DATA
# ============================================================
print("Loading data...")
with open(os.path.join(BASE_DIR, "minifigs.json")) as f:
    data = json.load(f)

valid_data = [d for d in data if d.get('img_local_path') and
              os.path.exists(os.path.join(BASE_DIR, d['img_local_path']))]
print(f"Valid images: {len(valid_data)}")

# ============================================================
# HELPER: build labels + split for a given merge strategy
# ============================================================
def build_split(valid_data, merge_map=None, town_split=False):
    """Apply merge, build cat2idx, reproduce the same stratified split."""
    import copy
    vd = copy.deepcopy(valid_data)

    for d in vd:
        if town_split and d['category'] == 'Town':
            d['merged_category'] = split_town(d['subcategory'])
        elif merge_map:
            d['merged_category'] = merge_map.get(d['category'], d['category'])
        else:
            d['merged_category'] = d['category']

    cat_key = 'merged_category'
    categories = [d[cat_key] for d in vd]
    cat_names = sorted(set(categories))
    cat2idx = {c: i for i, c in enumerate(cat_names)}
    idx2cat = {i: c for c, i in cat2idx.items()}
    num_classes = len(cat_names)

    labels = [cat2idx[d[cat_key]] for d in vd]
    label_counts = Counter(labels)
    small_idx = [i for i, l in enumerate(labels) if label_counts[l] < 7]
    big_idx = [i for i, l in enumerate(labels) if label_counts[l] >= 7]

    big_data = [vd[i] for i in big_idx]
    big_labels = [labels[i] for i in big_idx]

    _, temp_data, _, temp_labels = train_test_split(
        big_data, big_labels, test_size=0.3, random_state=42, stratify=big_labels)
    _, test_data, _, test_labels = train_test_split(
        temp_data, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)

    return test_data, test_labels, cat2idx, idx2cat, num_classes

# ============================================================
# DATASET
# ============================================================
class MinifigDataset(Dataset):
    def __init__(self, records, labels, transform, img_size):
        self.records, self.labels, self.transform, self.img_size = records, labels, transform, img_size
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        try: img = Image.open(os.path.join(BASE_DIR, self.records[idx]['img_local_path'])).convert('RGB')
        except: img = Image.new('RGB', (self.img_size, self.img_size), (128, 128, 128))
        return self.transform(img), self.labels[idx]

class PadToSquare:
    def __call__(self, img):
        w, h = img.size
        max_side = max(w, h)
        return ImageOps.pad(img, (max_side, max_side), color=(255, 255, 255))

# ============================================================
# EVALUATION FUNCTION
# ============================================================
def evaluate_model(model, test_loader, num_classes, idx2cat):
    model.eval()
    all_preds, all_labels_arr, all_probs = [], [], []
    with torch.no_grad():
        for images, lbl in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels_arr.extend(lbl.numpy())
            all_probs.extend(probs)

    all_preds = np.array(all_preds)
    all_labels_arr = np.array(all_labels_arr)
    all_probs = np.array(all_probs)

    acc = accuracy_score(all_labels_arr, all_preds)
    macro_f1 = f1_score(all_labels_arr, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_labels_arr, all_preds, average='weighted', zero_division=0)
    macro_prec = precision_score(all_labels_arr, all_preds, average='macro', zero_division=0)
    macro_rec = recall_score(all_labels_arr, all_preds, average='macro', zero_division=0)
    top3_acc = top_k_accuracy_score(all_labels_arr, all_probs, k=3, labels=range(num_classes))
    top5_acc = top_k_accuracy_score(all_labels_arr, all_probs, k=5, labels=range(num_classes))

    return {
        'Accuracy': acc, 'Top-3 Accuracy': top3_acc, 'Top-5 Accuracy': top5_acc,
        'Macro F1': macro_f1, 'Weighted F1': weighted_f1,
        'Macro Precision': macro_prec, 'Macro Recall': macro_rec
    }, all_preds, all_labels_arr, all_probs

def evaluate_with_tta(model, test_data, test_labels, num_classes, idx2cat, img_size, batch_size):
    """Run TTA evaluation for V2 model."""
    tta_transforms_list = [
        transforms.Compose([PadToSquare(), transforms.Resize((img_size, img_size)),
                            transforms.ToTensor(),
                            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
        transforms.Compose([PadToSquare(), transforms.Resize((img_size, img_size)),
                            transforms.RandomHorizontalFlip(p=1.0),
                            transforms.ToTensor(),
                            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
        transforms.Compose([PadToSquare(), transforms.Resize((img_size + 20, img_size + 20)),
                            transforms.CenterCrop(img_size), transforms.RandomRotation(10),
                            transforms.ToTensor(),
                            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
        transforms.Compose([PadToSquare(), transforms.Resize((img_size + 40, img_size + 40)),
                            transforms.CenterCrop(img_size),
                            transforms.ToTensor(),
                            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
        transforms.Compose([PadToSquare(), transforms.Resize((img_size, img_size)),
                            transforms.ColorJitter(brightness=0.15, contrast=0.15),
                            transforms.ToTensor(),
                            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    ]

    model.eval()
    # Get labels from first pass
    first_ds = MinifigDataset(test_data, test_labels, tta_transforms_list[0], img_size)
    first_loader = DataLoader(first_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    all_labels_arr = []
    all_probs_tta = None

    for t_idx, tta_t in enumerate(tta_transforms_list):
        ds = MinifigDataset(test_data, test_labels, tta_t, img_size)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
        probs_this = []
        labels_this = []
        with torch.no_grad():
            for images, lbl in loader:
                images = images.to(DEVICE)
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1).cpu().numpy()
                probs_this.extend(probs)
                if t_idx == 0:
                    labels_this.extend(lbl.numpy())
        if t_idx == 0:
            all_labels_arr = np.array(labels_this)
            all_probs_tta = np.array(probs_this)
        else:
            all_probs_tta += np.array(probs_this)
        print(f"  TTA view {t_idx+1}/{len(tta_transforms_list)} done")

    all_probs_tta /= len(tta_transforms_list)
    all_preds_tta = all_probs_tta.argmax(axis=1)

    acc = accuracy_score(all_labels_arr, all_preds_tta)
    macro_f1 = f1_score(all_labels_arr, all_preds_tta, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_labels_arr, all_preds_tta, average='weighted', zero_division=0)
    macro_prec = precision_score(all_labels_arr, all_preds_tta, average='macro', zero_division=0)
    macro_rec = recall_score(all_labels_arr, all_preds_tta, average='macro', zero_division=0)
    top3_acc = top_k_accuracy_score(all_labels_arr, all_probs_tta, k=3, labels=range(num_classes))
    top5_acc = top_k_accuracy_score(all_labels_arr, all_probs_tta, k=5, labels=range(num_classes))

    return {
        'Accuracy': acc, 'Top-3 Accuracy': top3_acc, 'Top-5 Accuracy': top5_acc,
        'Macro F1': macro_f1, 'Weighted F1': weighted_f1,
        'Macro Precision': macro_prec, 'Macro Recall': macro_rec
    }, all_preds_tta, all_labels_arr, all_probs_tta


# ============================================================
# 1. BASELINE — EfficientNet-B0, 122 classes, IMG_SIZE=128
# ============================================================
print("\n" + "="*80)
print("MODEL 1: BASELINE (EfficientNet-B0, 122 classes, 128px)")
print("="*80)

test_data_bl, test_labels_bl, cat2idx_bl, idx2cat_bl, nc_bl = build_split(valid_data)
print(f"Test set: {len(test_data_bl)} samples, {nc_bl} classes")

eval_tf_128 = transforms.Compose([
    transforms.Resize((128, 128)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_loader_bl = DataLoader(MinifigDataset(test_data_bl, test_labels_bl, eval_tf_128, 128),
                            batch_size=64, shuffle=False, num_workers=NUM_WORKERS)

model_bl = models.efficientnet_b0(weights=None)
model_bl.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model_bl.classifier[1].in_features, nc_bl))
model_bl.load_state_dict(torch.load(os.path.join(BASE_DIR, "baseline_results", "best_model.pth"),
                                     map_location=DEVICE, weights_only=True))
model_bl = model_bl.to(DEVICE)

metrics_bl, preds_bl, labels_bl, probs_bl = evaluate_model(model_bl, test_loader_bl, nc_bl, idx2cat_bl)
print(f"\n{'Metric':<30} {'Value':>10}")
print("-" * 42)
for k, v in metrics_bl.items():
    print(f"{k:<30} {v:>10.4f}")
del model_bl
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ============================================================
# 2. OPTION B — EfficientNet-B0, merged classes, IMG_SIZE=128
# ============================================================
print("\n" + "="*80)
print("MODEL 2: OPTION B (EfficientNet-B0, merged classes, 128px)")
print("="*80)

test_data_ob, test_labels_ob, cat2idx_ob, idx2cat_ob, nc_ob = build_split(valid_data, MERGE_MAP_B)
print(f"Test set: {len(test_data_ob)} samples, {nc_ob} classes")

test_loader_ob = DataLoader(MinifigDataset(test_data_ob, test_labels_ob, eval_tf_128, 128),
                            batch_size=64, shuffle=False, num_workers=NUM_WORKERS)

model_ob = models.efficientnet_b0(weights=None)
model_ob.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model_ob.classifier[1].in_features, nc_ob))
model_ob.load_state_dict(torch.load(os.path.join(BASE_DIR, "optionB_results", "best_model.pth"),
                                     map_location=DEVICE, weights_only=True))
model_ob = model_ob.to(DEVICE)

metrics_ob, preds_ob, labels_ob, probs_ob = evaluate_model(model_ob, test_loader_ob, nc_ob, idx2cat_ob)
print(f"\n{'Metric':<30} {'Value':>10}")
print("-" * 42)
for k, v in metrics_ob.items():
    print(f"{k:<30} {v:>10.4f}")
del model_ob
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ============================================================
# 3. OPTION B IMPROVED — EfficientNet-B0, merged, IMG_SIZE=224
# ============================================================
print("\n" + "="*80)
print("MODEL 3: OPTION B IMPROVED (EfficientNet-B0, merged, 224px, Focal+CutMix+MixUp)")
print("="*80)

test_data_oi, test_labels_oi, cat2idx_oi, idx2cat_oi, nc_oi = build_split(valid_data, MERGE_MAP_B)
print(f"Test set: {len(test_data_oi)} samples, {nc_oi} classes")

eval_tf_224 = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_loader_oi = DataLoader(MinifigDataset(test_data_oi, test_labels_oi, eval_tf_224, 224),
                            batch_size=32, shuffle=False, num_workers=NUM_WORKERS)

model_oi = models.efficientnet_b0(weights=None)
model_oi.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(model_oi.classifier[1].in_features, nc_oi))
model_oi.load_state_dict(torch.load(os.path.join(BASE_DIR, "optionB_improved_results", "best_model.pth"),
                                     map_location=DEVICE, weights_only=True))
model_oi = model_oi.to(DEVICE)

metrics_oi, preds_oi, labels_oi, probs_oi = evaluate_model(model_oi, test_loader_oi, nc_oi, idx2cat_oi)
print(f"\n{'Metric':<30} {'Value':>10}")
print("-" * 42)
for k, v in metrics_oi.items():
    print(f"{k:<30} {v:>10.4f}")
del model_oi
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ============================================================
# 4. OPTION B V2 — EfficientNet-B2, Town split, IMG_SIZE=260, TTA
# ============================================================
print("\n" + "="*80)
print("MODEL 4: OPTION B V2 (EfficientNet-B2, Town split, 260px, deeper head)")
print("="*80)

test_data_v2, test_labels_v2, cat2idx_v2, idx2cat_v2, nc_v2 = build_split(valid_data, MERGE_MAP_B, town_split=True)
print(f"Test set: {len(test_data_v2)} samples, {nc_v2} classes")

eval_tf_260 = transforms.Compose([
    PadToSquare(), transforms.Resize((260, 260)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_loader_v2 = DataLoader(MinifigDataset(test_data_v2, test_labels_v2, eval_tf_260, 260),
                            batch_size=24, shuffle=False, num_workers=NUM_WORKERS)

model_v2 = models.efficientnet_b2(weights=None)
in_features_v2 = model_v2.classifier[1].in_features  # 1408
model_v2.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features_v2, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, nc_v2)
)
model_v2.load_state_dict(torch.load(os.path.join(BASE_DIR, "optionB_v2_results", "best_model.pth"),
                                     map_location=DEVICE, weights_only=True))
model_v2 = model_v2.to(DEVICE)

# Standard eval (no TTA)
metrics_v2, preds_v2, labels_v2, probs_v2 = evaluate_model(model_v2, test_loader_v2, nc_v2, idx2cat_v2)
print(f"\n--- Without TTA ---")
print(f"{'Metric':<30} {'Value':>10}")
print("-" * 42)
for k, v in metrics_v2.items():
    print(f"{k:<30} {v:>10.4f}")

# TTA eval
print(f"\n--- With TTA (5 views) ---")
metrics_v2_tta, preds_v2_tta, labels_v2_tta, probs_v2_tta = evaluate_with_tta(
    model_v2, test_data_v2, test_labels_v2, nc_v2, idx2cat_v2, 260, 24)
print(f"\n{'Metric':<30} {'No TTA':>10} {'With TTA':>10} {'Delta':>8}")
print("-" * 60)
for k in metrics_v2:
    v1, v2t = metrics_v2[k], metrics_v2_tta[k]
    d = v2t - v1
    print(f"{k:<30} {v1:>10.4f} {v2t:>10.4f} {'+' if d>=0 else ''}{d:>7.4f}")

del model_v2
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ============================================================
# FULL COMPARISON TABLE
# ============================================================
print("\n" + "="*80)
print("FULL COMPARISON: ALL MODELS")
print("="*80)
print(f"{'Metric':<25} {'Baseline':>10} {'Opt B':>10} {'B Improv':>10} {'V2':>10} {'V2+TTA':>10}")
print("-" * 77)
for metric in metrics_bl:
    b = metrics_bl[metric]
    o = metrics_ob[metric]
    i = metrics_oi[metric]
    v = metrics_v2[metric]
    vt = metrics_v2_tta[metric]
    print(f"{metric:<25} {b:>10.4f} {o:>10.4f} {i:>10.4f} {v:>10.4f} {vt:>10.4f}")

print(f"{'Num Classes':<25} {nc_bl:>10} {nc_ob:>10} {nc_oi:>10} {nc_v2:>10} {nc_v2:>10}")

# ============================================================
# SAVE COMPARISON PLOT
# ============================================================
output_dir = os.path.join(BASE_DIR, "evaluation_results")
os.makedirs(output_dir, exist_ok=True)

# Bar chart comparison
metrics_to_plot = ['Accuracy', 'Top-3 Accuracy', 'Top-5 Accuracy', 'Macro F1', 'Weighted F1']
model_names = ['Baseline', 'Option B', 'B Improved', 'V2', 'V2+TTA']
all_metrics = [metrics_bl, metrics_ob, metrics_oi, metrics_v2, metrics_v2_tta]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(metrics_to_plot))
width = 0.15
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

for i, (name, mets) in enumerate(zip(model_names, all_metrics)):
    vals = [mets[m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, vals, width, label=name, color=colors[i])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics_to_plot)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "model_comparison.png"), dpi=150)
plt.close()

# Save results to text file
with open(os.path.join(output_dir, "comparison_results.txt"), 'w') as f:
    f.write("FULL MODEL COMPARISON\n")
    f.write("="*80 + "\n")
    f.write(f"{'Metric':<25} {'Baseline':>10} {'Opt B':>10} {'B Improv':>10} {'V2':>10} {'V2+TTA':>10}\n")
    f.write("-" * 77 + "\n")
    for metric in metrics_bl:
        b = metrics_bl[metric]
        o = metrics_ob[metric]
        i = metrics_oi[metric]
        v = metrics_v2[metric]
        vt = metrics_v2_tta[metric]
        f.write(f"{metric:<25} {b:>10.4f} {o:>10.4f} {i:>10.4f} {v:>10.4f} {vt:>10.4f}\n")
    f.write(f"{'Num Classes':<25} {nc_bl:>10} {nc_ob:>10} {nc_oi:>10} {nc_v2:>10} {nc_v2:>10}\n")

print(f"\nComparison chart saved to: {os.path.join(output_dir, 'model_comparison.png')}")
print(f"Results saved to: {os.path.join(output_dir, 'comparison_results.txt')}")
print("\nDone!")


## Cleanup Policy

The repository keeps model code in this notebook. Generated assets such as checkpoints, result folders, local data, images, logs, and temporary notebooks stay out of Git via `.gitignore`.